# 05 — Final Load Preparation

**Purpose:** Produce clean, renamed, validated CSV files ready to connect directly into Tableau.
One file per dashboard. No analysis happens here — this is pure data plumbing.

**Rules applied in every table:**
- All column names are converted to **PascalCase** for Tableau readability.
- All output files are written to `data/production/`.
- Every file is validated against a list of required columns before export.
- The notebook ends with a Tableau connection guide.

**Input:** 8 joined tables in `data/joined/` (J1–J8), produced by `join_tables.py`.

**Output:** 11 production CSV files + 1 master flat table.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
JOINED = PROJECT_ROOT / 'data' / 'joined'
PROD   = PROJECT_ROOT / 'data' / 'production'
PROD.mkdir(parents=True, exist_ok=True)

# ── Helper functions ───────────────────────────────────────────
def save(df, name):
    path = PROD / f'{name}.csv'
    df.to_csv(path, index=False)
    null_pct = df.isnull().mean().mean() * 100
    print(f'  Saved: {name}.csv   {len(df):,} rows  {len(df.columns)} cols  overall_null%={null_pct:.1f}')

def validate(df, name, required_columns):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        print(f'  [WARN] {name} — missing columns: {missing}')
    else:
        print(f'  [OK] {name} — all required columns present')
    high_null = {c: round(df[c].isnull().mean()*100, 1)
                 for c in df.columns if df[c].isnull().mean() > 0.3}
    if high_null:
        print(f'  [WARN] Columns >30% null: {high_null}')

# ── Load all 8 joined tables ───────────────────────────────────
J1 = pd.read_csv(JOINED / 'J1_master_race_results.csv', low_memory=False)
J2 = pd.read_csv(JOINED / 'J2_driver_alpha.csv')
J3 = pd.read_csv(JOINED / 'J3_qualifying_analysis.csv')
J4 = pd.read_csv(JOINED / 'J4_championship_progression.csv')
J5 = pd.read_csv(JOINED / 'J5_pit_strategy.csv')
J6 = pd.read_csv(JOINED / 'J6_lap_by_lap.csv', low_memory=False)
J7 = pd.read_csv(JOINED / 'J7_reliability_dnf.csv')
J8 = pd.read_csv(JOINED / 'J8_constructor_reliability.csv')

print('All 8 joined tables loaded successfully.')
for name, df in [('J1',J1),('J2',J2),('J3',J3),('J4',J4),('J5',J5),('J6',J6),('J7',J7),('J8',J8)]:
    print(f'  {name}: {len(df):>8,} rows × {len(df.columns)} cols')

All 8 joined tables loaded successfully.
  J1:   23,777 rows × 46 cols
  J2:    3,397 rows × 24 cols
  J3:    7,516 rows × 31 cols
  J4:   23,777 rows × 20 cols
  J5:    2,782 rows × 31 cols
  J6:  426,633 rows × 21 cols
  J7:   23,777 rows × 28 cols
  J8:    1,041 rows × 18 cols


---
## Dashboard 1 — Driver Alpha (Pillar A)

**What this dashboard shows:**
- Scatter plot of Driver Alpha vs Points-Per-Race (who outperforms their car?)
- Bar chart of top 20 drivers by career alpha
- Line chart of alpha evolution by season for selected drivers
- Heatmap of driver × constructor performance pairings

**Tables produced:** `DB1_driver_alpha_season.csv`, `DB1_driver_career_summary.csv`

In [2]:
# ══════════════════════════════════════════════════════════════
# Table 1: DB1_driver_alpha_season.csv  (from J2)
# ══════════════════════════════════════════════════════════════

db1_season = J2.rename(columns={
    'full_name':            'Driver',
    'constructor_name':     'Constructor',
    'year':                 'Season',
    'era':                  'Era',
    'season_points':        'SeasonPoints',
    'races':                'RacesThisSeason',
    'wins':                 'SeasonWins',
    'podiums':              'SeasonPodiums',
    'dnfs':                 'SeasonDNFs',
    'avg_finish_pos':       'AvgFinishPos',
    'avg_grid_pos':         'AvgGridPos',
    'avg_grid_delta':       'AvgGridDelta',
    'ppr':                  'PointsPerRace',
    'constructor_avg_ppr':  'ConstructorAvgPPR',
    'driver_alpha':         'DriverAlpha',
    'career_alpha':         'CareerAlpha',
    'win_rate':             'WinRate',
    'dnf_rate':             'DNFRate',
    'total_races':          'CareerTotalRaces',
}).copy()

validate(db1_season, 'DB1_driver_alpha_season',
         ['Driver', 'Season', 'Constructor', 'Era', 'DriverAlpha', 'PointsPerRace'])
save(db1_season, 'DB1_driver_alpha_season')

# ══════════════════════════════════════════════════════════════
# Table 2: DB1_driver_career_summary.csv  (from J1 + J2 + J3)
# ══════════════════════════════════════════════════════════════

career = J1.groupby(['driverId', 'full_name']).agg(
    TotalRaces      = ('raceId',         'count'),
    TotalPoints     = ('points',         'sum'),
    TotalWins       = ('is_win',         'sum'),
    TotalPodiums    = ('is_podium',      'sum'),
    TotalDNFs       = ('is_dnf',         'sum'),
    AvgFinishPos    = ('positionOrder',   'mean'),
    AvgGridDelta    = ('grid_vs_finish',  'mean'),
).reset_index()

career['PPR']        = career['TotalPoints'] / career['TotalRaces']
career['WinRate']    = career['TotalWins']    / career['TotalRaces']
career['PodiumRate'] = career['TotalPodiums'] / career['TotalRaces']
career['DNFRate']    = career['TotalDNFs']    / career['TotalRaces']

# Merge career alpha from J2
alpha_career = J2.groupby('driverId')['driver_alpha'].mean().reset_index()
alpha_career.rename(columns={'driver_alpha': 'CareerAlpha'}, inplace=True)
career = career.merge(alpha_career, on='driverId', how='left')

# Merge qualifying stats from J3
q3_valid = J3.dropna(subset=['q1_vs_team_avg_ms'])
q_stats = q3_valid.groupby('driverId').agg(
    q_sessions = ('q1_vs_team_avg_ms', 'count'),
    q1_faster  = ('q1_vs_team_avg_ms', lambda x: (x < 0).sum()),
    avg_q1_ms  = ('q1_ms',             'median'),
).reset_index()
q_stats = q_stats[q_stats['q_sessions'] >= 30].copy()
q_stats['QualWinPct'] = q_stats['q1_faster'] / q_stats['q_sessions'] * 100
q_stats.rename(columns={'avg_q1_ms': 'MedianQ1Ms'}, inplace=True)
career = career.merge(q_stats[['driverId', 'QualWinPct', 'MedianQ1Ms']], on='driverId', how='left')

career.rename(columns={'full_name': 'Driver'}, inplace=True)

validate(career, 'DB1_driver_career_summary',
         ['Driver', 'CareerAlpha', 'WinRate', 'TotalRaces'])
save(career, 'DB1_driver_career_summary')

  [OK] DB1_driver_alpha_season — all required columns present
  Saved: DB1_driver_alpha_season.csv   3,397 rows  24 cols  overall_null%=0.3
  [OK] DB1_driver_career_summary — all required columns present
  [WARN] Columns >30% null: {'QualWinPct': 91.9, 'MedianQ1Ms': 91.9}
  Saved: DB1_driver_career_summary.csv   841 rows  16 cols  overall_null%=12.0


---
## Dashboard 2 — Pit Strategy (Pillar B)

**Scope:** 2011–2017 only (pit stop data availability).

**What this dashboard shows:**
- Strategy distribution (1-stop vs 2-stop vs 3-stop) by era/circuit
- Pit stop efficiency: median pit time vs finish position
- Undercut/overcut analysis via lap-by-lap position changes around pit laps
- First stop timing impact on race outcome

**Tables produced:** `DB2_pit_strategy.csv`, `DB2_lap_by_lap.csv`

In [3]:
# ══════════════════════════════════════════════════════════════
# Table 1: DB2_pit_strategy.csv  (from J5)
# ══════════════════════════════════════════════════════════════

db2_pit = J5.rename(columns={
    'full_name':             'Driver',
    'constructor_name':      'Constructor',
    'race_name':             'RaceName',
    'year':                  'Season',
    'era':                   'Era',
    'country':               'Country',
    'circuit_name':          'Circuit',
    'total_stops':           'TotalStops',
    'strategy':              'Strategy',
    'median_pit_sec':        'MedianPitSec',
    'total_pit_sec':         'TotalPitSec',
    'first_pit_lap':         'FirstPitLap',
    'last_pit_lap':          'LastPitLap',
    'positionOrder':         'FinishPosition',
    'points':                'Points',
    'grid':                  'GridPosition',
    'laps':                  'TotalLaps',
    'dnf':                   'DNF',
    'grid_vs_finish':        'GridDelta',
    'status':                'Status',
    'dnf_category':          'DNFCategory',
    'pit_time_per_stop_sec': 'AvgSecPerStop',
}).copy()

# Derived: FirstStopPct — how early in the race was the first stop?
db2_pit['FirstStopPct'] = np.where(
    db2_pit['TotalLaps'] > 0,
    db2_pit['FirstPitLap'] / db2_pit['TotalLaps'] * 100,
    np.nan
)

validate(db2_pit, 'DB2_pit_strategy',
         ['Driver', 'Season', 'TotalStops', 'Strategy', 'FinishPosition', 'MedianPitSec'])
save(db2_pit, 'DB2_pit_strategy')

# ══════════════════════════════════════════════════════════════
# Table 2: DB2_lap_by_lap.csv  (from J6, racing laps only)
# ══════════════════════════════════════════════════════════════

db2_lap = J6[J6['is_racing_lap'] == True].rename(columns={
    'full_name':        'Driver',
    'constructor_name': 'Constructor',
    'race_name':        'RaceName',
    'year':             'Season',
    'era':              'Era',
    'lap_time_sec':     'LapTimeSec',
    'position':         'TrackPosition',
    'position_change':  'PositionChange',
    'pit_stop_lap':     'IsPitLap',
    'pit_duration_ms':  'PitDurationMs',
}).copy()

validate(db2_lap, 'DB2_lap_by_lap',
         ['Driver', 'Season', 'lap', 'LapTimeSec', 'TrackPosition'])
save(db2_lap, 'DB2_lap_by_lap')

  [OK] DB2_pit_strategy — all required columns present
  Saved: DB2_pit_strategy.csv   2,782 rows  32 cols  overall_null%=0.0
  [OK] DB2_lap_by_lap — all required columns present
  [WARN] Columns >30% null: {'PitDurationMs': 98.5, 'is_valid_stop': 98.5}
  Saved: DB2_lap_by_lap.csv   426,296 rows  21 cols  overall_null%=9.6


---
## Dashboard 3 — Reliability & Risk (Pillar C)

**What this dashboard shows:**
- DNF rate by era (Pre-V10 → V10 → V8 → Hybrid): did reliability improve?
- Mechanical vs accident vs other DNF breakdown
- Constructor reliability index over time (rolling 3-season average)
- Points lost to DNFs — the hidden cost of unreliability

**Tables produced:** `DB3_reliability_dnf.csv`, `DB3_constructor_reliability.csv`

In [4]:
# ══════════════════════════════════════════════════════════════
# Table 1: DB3_reliability_dnf.csv  (from J7)
# ══════════════════════════════════════════════════════════════

db3_dnf = J7.rename(columns={
    'full_name':                'Driver',
    'constructor_name':         'Constructor',
    'nationality_driver':       'DriverNationality',
    'race_name':                'RaceName',
    'year':                     'Season',
    'era':                      'Era',
    'country':                  'Country',
    'circuit_name':             'Circuit',
    'grid':                     'GridPosition',
    'positionOrder':            'FinishPosition',
    'laps':                     'LapsCompleted',
    'total_race_laps':          'TotalRaceLaps',
    'laps_pct_completed':       'LapsPctCompleted',
    'status':                   'Status',
    'dnf_category':             'DNFCategory',
    'is_dnf':                   'IsDNF',
    'is_win':                   'IsWin',
    'is_podium':                'IsPodium',
    'potential_points_lost':    'PotentialPointsLost',
}).copy()

validate(db3_dnf, 'DB3_reliability_dnf',
         ['Driver', 'Season', 'Era', 'DNFCategory', 'IsDNF'])
save(db3_dnf, 'DB3_reliability_dnf')

# ══════════════════════════════════════════════════════════════
# Table 2: DB3_constructor_reliability.csv  (from J8)
# ══════════════════════════════════════════════════════════════

db3_rel = J8.rename(columns={
    'constructor_name':      'Constructor',
    'year':                  'Season',
    'era':                   'Era',
    'entries':               'Entries',
    'total_dnfs':            'TotalDNFs',
    'mechanical_dnfs':       'MechanicalDNFs',
    'accident_dnfs':         'AccidentDNFs',
    'other_dnfs':            'OtherDNFs',
    'total_points':          'TotalPoints',
    'total_wins':            'TotalWins',
    'pts_lost_to_dnf':       'PtsLostToDNF',
    'dnf_rate':              'DNFRate',
    'mechanical_dnf_rate':   'MechDNFRate',
    'reliability_index':     'ReliabilityIndex',
    'rolling3_reliability':  'Rolling3Reliability',
    'era_reliability':       'EraReliability',
    'era_mech_dnf_rate':     'EraMechDNFRate',
}).copy()

validate(db3_rel, 'DB3_constructor_reliability',
         ['Constructor', 'Season', 'Era', 'ReliabilityIndex', 'DNFRate'])
save(db3_rel, 'DB3_constructor_reliability')

  [OK] DB3_reliability_dnf — all required columns present
  Saved: DB3_reliability_dnf.csv   23,777 rows  28 cols  overall_null%=0.0
  [OK] DB3_constructor_reliability — all required columns present
  Saved: DB3_constructor_reliability.csv   1,041 rows  18 cols  overall_null%=0.0


---
## Dashboard 4 — Championship Story (Pillar A + Dashboard 4)

**What this dashboard shows:**
- Bump chart of championship position by round (who led when?)
- Cumulative points line chart for the top contenders each season
- Gap-to-leader area chart showing how close the fight was
- Geographic map of circuits with race count and winner statistics

**Tables produced:** `DB4_championship_story.csv`, `DB4_circuit_map.csv`

In [5]:
# ══════════════════════════════════════════════════════════════
# Table 1: DB4_championship_story.csv  (from J4)
# ══════════════════════════════════════════════════════════════

db4_champ = J4.rename(columns={
    'full_name':            'Driver',
    'constructor_name':     'Constructor',
    'race_name':            'RaceName',
    'year':                 'Season',
    'era':                  'Era',
    'country':              'Country',
    'round':                'Round',
    'points':               'Points',
    'is_win':               'IsWin',
    'is_podium':            'IsPodium',
    'is_dnf':               'IsDNF',
    'positionOrder':        'FinishPosition',
    'grid':                 'GridPosition',
    'cumulative_points':    'CumulativePoints',
    'cumulative_wins':      'CumulativeWins',
    'championship_pos':     'ChampionshipPosition',
    'gap_to_leader':        'GapToLeader',
}).copy()

validate(db4_champ, 'DB4_championship_story',
         ['Driver', 'Season', 'Round', 'CumulativePoints', 'ChampionshipPosition', 'GapToLeader'])
save(db4_champ, 'DB4_championship_story')

# ══════════════════════════════════════════════════════════════
# Table 2: DB4_circuit_map.csv  (from J1)
# ══════════════════════════════════════════════════════════════

# One row per circuit for geographic map visualization
circuit_map = J1.groupby(['circuitId', 'circuit_name', 'country', 'lat', 'lng']).agg(
    TotalRaces     = ('raceId', 'nunique'),
    FirstYear      = ('year',   'min'),
    LastYear       = ('year',   'max'),
).reset_index()

# Average points scored by the race winner at each circuit
winners = J1[J1['is_win'] == 1]
avg_winner_pts = winners.groupby('circuitId')['points'].mean().reset_index()
avg_winner_pts.rename(columns={'points': 'AvgPointsWinner'}, inplace=True)
circuit_map = circuit_map.merge(avg_winner_pts, on='circuitId', how='left')

validate(circuit_map, 'DB4_circuit_map',
         ['circuit_name', 'country', 'lat', 'lng'])
save(circuit_map, 'DB4_circuit_map')

  [OK] DB4_championship_story — all required columns present
  Saved: DB4_championship_story.csv   23,777 rows  20 cols  overall_null%=0.0
  [OK] DB4_circuit_map — all required columns present
  Saved: DB4_circuit_map.csv   72 rows  9 cols  overall_null%=0.0


---
## Master Flat Table

The single richest table in the project. Built by joining J1 (base) with qualifying,
championship, and driver-alpha data. Use this for ad-hoc Tableau analysis when you
need everything in one place without configuring joins inside Tableau.

**Expected shape:** ~23,777 rows × 55–60 columns.

In [6]:
# ══════════════════════════════════════════════════════════════
# MASTER_tableau_ready.csv  (J1 + J3 + J4 + J2)
# ══════════════════════════════════════════════════════════════

master = J1.copy()

# ── Left merge qualifying data from J3 ─────────────────────────
q_cols = ['raceId', 'driverId', 'q1_ms', 'q2_ms', 'q3_ms',
          'best_qual_ms', 'q1_vs_team_avg_ms', 'reached_q2', 'reached_q3']
master = master.merge(J3[q_cols], on=['raceId', 'driverId'], how='left')

# ── Left merge championship progression from J4 ────────────────
# Deduplicate on (raceId, driverId) — 89 rows duplicated from mid-season team changes
c_cols = ['raceId', 'driverId', 'cumulative_points', 'championship_pos', 'gap_to_leader']
c_sub = J4[c_cols].drop_duplicates(subset=['raceId', 'driverId'], keep='first')
master = master.merge(c_sub, on=['raceId', 'driverId'], how='left')

# ── Left merge driver alpha from J2 ────────────────────────────
# J2 has one row per driver × year × constructor; join on all three to avoid row inflation
a_cols = J2[['driverId', 'year', 'constructorId', 'driver_alpha', 'constructor_avg_ppr', 'ppr']].copy()
master = master.merge(a_cols, on=['driverId', 'year', 'constructorId'], how='left')

print(f'Master table shape: {master.shape}')
validate(master, 'MASTER_tableau_ready',
         ['full_name', 'year', 'era', 'points', 'is_win', 'is_dnf',
          'dnf_category', 'driver_alpha', 'championship_pos', 'q1_ms'])
save(master, 'MASTER_tableau_ready')

Master table shape: (23777, 59)
  [OK] MASTER_tableau_ready — all required columns present
  [WARN] Columns >30% null: {'q1_ms': 68.9, 'q2_ms': 84.6, 'q3_ms': 90.8, 'best_qual_ms': 68.9, 'q1_vs_team_avg_ms': 68.9, 'reached_q2': 68.4, 'reached_q3': 68.4}
  Saved: MASTER_tableau_ready.csv   23,777 rows  59 cols  overall_null%=8.9


In [7]:
# ══════════════════════════════════════════════════════════════
# Final Validation — Production File Inventory
# ══════════════════════════════════════════════════════════════

print('\n' + '═' * 75)
print('  PRODUCTION FILE INVENTORY')
print('═' * 75)
print(f'{"File":<50}  {"Rows":>8}  {"Cols":>4}  {"Size":>7}')
print('─' * 75)

for f in sorted(PROD.glob('*.csv')):
    df = pd.read_csv(f)
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'{f.name:<50}  {len(df):>8,}  {len(df.columns):>4}  {size_mb:>6.1f}M')

print('─' * 75)
print('All production files written successfully.')
print(f'Output directory: {PROD}')


═══════════════════════════════════════════════════════════════════════════
  PRODUCTION FILE INVENTORY
═══════════════════════════════════════════════════════════════════════════
File                                                    Rows  Cols     Size
───────────────────────────────────────────────────────────────────────────
DB1_driver_alpha_season.csv                            3,397    24     0.6M
DB1_driver_career_summary.csv                            841    16     0.1M


/var/folders/jv/5k5vjh9539b3tbbmd8mfb6n80000gn/T/ipykernel_80763/3145967958.py:12: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


DB2_lap_by_lap.csv                                   426,296    21    50.4M
DB2_pit_strategy.csv                                   2,782    32     0.6M
DB3_constructor_reliability.csv                        1,041    18     0.1M
DB3_reliability_dnf.csv                               23,777    28     4.3M
DB4_championship_story.csv                            23,777    20     2.4M
DB4_circuit_map.csv                                       72     9     0.0M
MASTER_tableau_ready.csv                              23,777    59     7.9M
───────────────────────────────────────────────────────────────────────────
All production files written successfully.
Output directory: /Users/aryankumar/Desktop/Code/DVA-CAP/Section_E_Group_3_PitLane/data/production


---
## Tableau Connection Guide

---

### Dashboard 1 — Driver Alpha

| Item | Detail |
|------|--------|
| **Primary source** | `DB1_driver_alpha_season.csv` |
| **Secondary source** | `DB1_driver_career_summary.csv` |
| **Join key** | `driverId` (integer key, present in both) |
| **Key dimensions (filters)** | Era, Season, Constructor |
| **Key measures** | DriverAlpha, PointsPerRace, WinRate, AvgGridDelta |
| **Recommended charts** | Scatter (DriverAlpha vs PointsPerRace), Bar (top 20 drivers), Line (alpha by season per driver), Heatmap (driver × constructor) |

---

### Dashboard 2 — Pit Strategy

| Item | Detail |
|------|--------|
| **Primary source** | `DB2_pit_strategy.csv` (2011–2017 only) |
| **Secondary source** | `DB2_lap_by_lap.csv` (2011–2017 only, ~426k rows — use as detail table, not primary) |
| **Key dimensions** | Strategy, Constructor, Season, Circuit |
| **Key measures** | MedianPitSec, TotalStops, FinishPosition, Points, FirstStopPct |
| **Note** | Filter `DNF = False` for most charts. Include DNFs only when analysing failure causes. |

---

### Dashboard 3 — Reliability & Risk

| Item | Detail |
|------|--------|
| **Primary source** | `DB3_reliability_dnf.csv` |
| **Secondary source** | `DB3_constructor_reliability.csv` |
| **Join key** | `constructorId` + `Season` (both present in both tables) |
| **Key dimensions** | Era, DNFCategory, Constructor |
| **Key measures** | ReliabilityIndex, DNFRate, MechDNFRate, PotentialPointsLost |
| **Recommended charts** | Stacked bar (DNF rate by era), Heatmap (reliability over time), Bar (points lost) |

---

### Dashboard 4 — Championship Story

| Item | Detail |
|------|--------|
| **Primary source** | `DB4_championship_story.csv` |
| **Map source** | `DB4_circuit_map.csv` (use `lat` and `lng` for Tableau geographic map; `TotalRaces` for bubble size) |
| **Key dimensions** | Season, Round, Driver, Constructor |
| **Key measures** | CumulativePoints, ChampionshipPosition, GapToLeader, CumulativeWins |
| **Recommended charts** | Bump chart (championship position by round), Line (cumulative points), Area (gap to leader) |

---

### Ad-hoc Analysis

Use **`MASTER_tableau_ready.csv`** — it contains everything in one flat table.
It is the slowest to load (~23k rows, ~60 columns) but requires **no joins** in Tableau.